### Configuración del entorno y verificación de integridad de los datos crudos

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import openpyxl

# Agregar la raíz del proyecto al path para importar desde src/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.auxiliar_functions import (
    RAW_DATA_DIR,
    DATA_DIR,
    SEMILLA,
    huella,
    verificar_checksums,
)

# 1. Verificar integridad SHA-256 de los 3 archivos Excel crudos
verificar_checksums()

Verificación de integridad completada: 3 archivo(s) coinciden.


True

### Comparar las hojas existentes entre los 3 archivos Excel

In [2]:
# Rutas de los 3 archivos Excel en data/raw/
ruta_2020_2022 = RAW_DATA_DIR / "Totalizadores Planta 2020_2022.xlsx"
ruta_2021_2023 = RAW_DATA_DIR / "Totalizadores Planta - 2021_2023.xlsx"
ruta_2022_2023 = RAW_DATA_DIR / "Totalizadores Planta 2022_2023.xlsx"

# Consultamos los nombres de las hojas de cada archivo con pandas
hojas_2020_2022 = pd.ExcelFile(ruta_2020_2022).sheet_names
hojas_2021_2023 = pd.ExcelFile(ruta_2021_2023).sheet_names
hojas_2022_2023 = pd.ExcelFile(ruta_2022_2023).sheet_names

print("Cantidad de hojas en 2020_2022:", len(hojas_2020_2022))
print("Cantidad de hojas en 2021_2023:", len(hojas_2021_2023))
print("Cantidad de hojas en 2022_2023:", len(hojas_2022_2023))

print("\nLista de hojas en 2020_2022:")
print(hojas_2020_2022)

# Vemos que hoja tiene de mas el archivo 2021_2023 respecto a los otros dos
hoja_diferente = set(hojas_2021_2023) - set(hojas_2020_2022)
print("\nHoja que solo existe en 2021_2023:", hoja_diferente)

Cantidad de hojas en 2020_2022: 15
Cantidad de hojas en 2021_2023: 16
Cantidad de hojas en 2022_2023: 15

Lista de hojas en 2020_2022:
['Consolidado KPI', 'Consolidado Produccion', 'Totalizadores Produccion', 'Consolidado EE', 'Totalizadores Energia', 'Consolidado Agua', 'Totalizadores Agua', 'Consolidado GasVapor', 'Totalizadores Gas y Vapor', 'Consolidado Aire', 'Totalizadores Aire', 'Totalizadores CO2', 'Totalizadores Efluentes', 'Totalizadores Glicol', 'Auxiliar']

Hoja que solo existe en 2021_2023: {'Seguimiento Dia'}


# Fase 1.1

### Vemos el solapamiento de dias

Esta celda carga la hoja principal Consolidado EE (donde está nuestro target Frio (Kw)) de los 3 archivos por separado. Luego hace lo que pide la 1.1.
- Rango de fechas y filas con DIA nulo en cada archivo.
- Qué fechas tienen más de 25 registros horarios dentro de un mismo archivo (jornadas apiladas).
- Cuántas fechas se solapan entre archivos y si el valor de Frio (Kw) al final del día coincide o difiere entre ellos
- Cuántos días tienen registro a las 23:00 vs. 23:59 y cuál es su diferencia promedio en kW.

In [3]:
# Cargamos la hoja 'Consolidado EE' de cada archivo en 3 DataFrames separados
df_ee_1 = pd.read_excel(ruta_2020_2022, sheet_name="Consolidado EE")
df_ee_2 = pd.read_excel(ruta_2021_2023, sheet_name="Consolidado EE")
df_ee_3 = pd.read_excel(ruta_2022_2023, sheet_name="Consolidado EE")

# Los guardamos en un diccionario con nombres claros
archivos_ee = {
    "2020_2022": df_ee_1,
    "2021_2023": df_ee_2,
    "2022_2023": df_ee_3,
}

for nombre, df in archivos_ee.items():
    # Convertimos la columna DIA a fecha (los valores invalidos quedan como NaT / nulos)
    df["DIA"] = pd.to_datetime(df["DIA"], errors="coerce")

    # Contamos cuantas filas tiene cada fecha dentro de este archivo
    filas_por_fecha = df["DIA"].value_counts()
    fechas_con_mas_de_25 = filas_por_fecha[filas_por_fecha > 25]

    print("--------------------------------------------------")
    print("Archivo:", nombre)
    print("  Filas totales:", len(df), "| Filas sin fecha valida:", df["DIA"].isna().sum())
    print("  Fecha inicial:", df["DIA"].min().date(), "| Fecha final:", df["DIA"].max().date())
    print("  Cantidad de fechas unicas:", df["DIA"].nunique())
    print("  Cantidad de fechas con mas de 25 registros:", len(fechas_con_mas_de_25))

    if len(fechas_con_mas_de_25) > 0:
        print("  Detalle de fechas con mas de 25 registros:")
        print(fechas_con_mas_de_25.sort_index())

--------------------------------------------------
Archivo: 2020_2022
  Filas totales: 16054 | Filas sin fecha valida: 479
  Fecha inicial: 2020-07-01 | Fecha final: 2022-03-16
  Cantidad de fechas unicas: 612
  Cantidad de fechas con mas de 25 registros: 8
  Detalle de fechas con mas de 25 registros:
DIA
2020-09-30     50
2020-11-30     50
2021-02-28    100
2021-04-30     50
2021-06-30     51
2021-09-30     50
2021-11-30     50
2022-02-28    100
Name: count, dtype: int64
--------------------------------------------------
Archivo: 2021_2023
  Filas totales: 15450 | Filas sin fecha valida: 134
  Fecha inicial: 2021-07-01 | Fecha final: 2023-03-06
  Cantidad de fechas unicas: 602
  Cantidad de fechas con mas de 25 registros: 8
  Detalle de fechas con mas de 25 registros:
DIA
2021-09-30     50
2021-11-30     50
2022-02-28    100
2022-04-30     50
2022-06-30     51
2022-09-30     50
2022-11-30     50
2023-02-28     99
Name: count, dtype: int64
----------------------------------------------

Por que hay fechas con 50 o 100 reigstros?:
- 30 de abril, 30 de junio, 30 de septiembre, 30 de noviembre  tienen 50 filas(2×25). Son exactamente los 4 meses del año que tienen 30 días en lugar de 31.
- 28 de febrero: tiene 100 filas (4x25). Es el unico mes que tiene 28 dias, le faltan los dias 29,30,31 y 3 dias x 25hs =75 filas extras.

Entiendo que la explicacion de esto es que quien armo la planilla de excel en la plana, diseño todos los meses como si tuvieran 31 dias de 25hs. En los meses de 30 dias, el dia 31 quedo etiquetado con la fecha del dia 30... Es una suposicion del problema inicial del dataset.

In [4]:
# Vemos que fechas aparecen en el Excel justo antes y justo despues del 2020-09-30
fechas_sept_oct = df_ee_1[(df_ee_1["DIA"] >= "2020-09-28") & (df_ee_1["DIA"] <= "2020-10-03")]
print(fechas_sept_oct.groupby("DIA").size())

DIA
2020-09-28    25
2020-09-29    25
2020-09-30    50
2020-10-01    25
2020-10-02    25
2020-10-03    25
dtype: int64


Efectivamente, no esta el 31 de septiembre

### Vemos que hay dentro de los dias con mas de 25 registros

me fijo las primeras 25 filas contra las segundas 25 filas del 2020-09-30 para sabner cuales son las lecturas reales de ese dia y cuales son las filas sobrantes


In [5]:
# Filtramos el dia 2020-09-30 del primer archivo (que vimos que tiene 50 filas)
ejemplo_dia_apilado = df_ee_1[df_ee_1["DIA"] == "2020-09-30"][["DIA", "HORA", "Planta (Kw)", "Frio (Kw)"]]

print("Primer bloque de 25 filas del 2020-09-30 (filas 0 a 24):")
print("  Ultimo registro del bloque 1 (fila 24):")
print(ejemplo_dia_apilado.iloc[23:25])

print("\nSegundo bloque de 25 filas del 2020-09-30 (filas 25 a 49):")
print("  Primeros y ultimos registros del bloque 2:")
print(ejemplo_dia_apilado.iloc[25:28])
print(ejemplo_dia_apilado.iloc[48:50])

Primer bloque de 25 filas del 2020-09-30 (filas 0 a 24):
  Ultimo registro del bloque 1 (fila 24):
            DIA      HORA  Planta (Kw)  Frio (Kw)
2222 2020-09-30  23:00:00     77968.45    28609.0
2223 2020-09-30  23:59:00     80872.30    29623.0

Segundo bloque de 25 filas del 2020-09-30 (filas 25 a 49):
  Primeros y ultimos registros del bloque 2:
            DIA      HORA  Planta (Kw)  Frio (Kw)
2224 2020-09-30  00:00:00         0.00        0.0
2225 2020-09-30  01:00:00      3020.88     1052.0
2226 2020-09-30  02:00:00      5778.28     1821.0
            DIA      HORA  Planta (Kw)  Frio (Kw)
2247 2020-09-30  23:00:00     70588.07    24829.0
2248 2020-09-30  23:59:00     73127.29    25558.0


### Cuantifico las fechas solapadas entre los archivos

In [6]:
#  Obtenemos el conjunto de fechas unicas (sin nulos) de cada uno de los 3 archivos
fechas_archivo_1 = set(df_ee_1["DIA"].dropna())
fechas_archivo_2 = set(df_ee_2["DIA"].dropna())
fechas_archivo_3 = set(df_ee_3["DIA"].dropna())

# Calculamos las fechas en comun (interseccion) entre pares de archivos
solapadas_1_y_2 = sorted(fechas_archivo_1.intersection(fechas_archivo_2))
solapadas_2_y_3 = sorted(fechas_archivo_2.intersection(fechas_archivo_3))
solapadas_1_y_3 = sorted(fechas_archivo_1.intersection(fechas_archivo_3))

print("1. CANTIDAD DE FECHAS SOLAPADAS ENTRE ARCHIVOS:")
print("  - Entre 2020_2022 y 2021_2023:", len(solapadas_1_y_2), "fechas")
print("  - Entre 2021_2023 y 2022_2023:", len(solapadas_2_y_3), "fechas")
print("  - Entre 2020_2022 y 2022_2023:", len(solapadas_1_y_3), "fechas")

#  Funcion para obtener el valor de 'Frio (Kw)' a las 23:59 del primer bloque de cada dia
def obtener_cierre_2359(df):
    df_copia = df.dropna(subset=["DIA"]).copy()
    df_copia["HORA_texto"] = df_copia["HORA"].astype(str).str.strip()
    df_copia["Frio (Kw)"] = pd.to_numeric(df_copia["Frio (Kw)"], errors="coerce")
    # Filtramos las 23:59 y usamos .first() para quedarnos con las primeras 25 horas del dia
    cierre = df_copia[df_copia["HORA_texto"].str.startswith("23:59")].groupby("DIA")["Frio (Kw)"].first()
    return cierre

cierre_1 = obtener_cierre_2359(df_ee_1)
cierre_2 = obtener_cierre_2359(df_ee_2)
cierre_3 = obtener_cierre_2359(df_ee_3)

# Usamos .reindex() en lugar de .loc[] para que si a una fecha le falta la hora 23:59
# coloque NaN en vez de lanzar KeyError
comparacion_1_2 = pd.DataFrame({
    "frio_2020_2022": cierre_1.reindex(solapadas_1_y_2),
    "frio_2021_2023": cierre_2.reindex(solapadas_1_y_2)
})
diferencia_1_2 = (comparacion_1_2["frio_2020_2022"] - comparacion_1_2["frio_2021_2023"]).abs()

comparacion_2_3 = pd.DataFrame({
    "frio_2021_2023": cierre_2.reindex(solapadas_2_y_3),
    "frio_2022_2023": cierre_3.reindex(solapadas_2_y_3)
})
diferencia_2_3 = (comparacion_2_3["frio_2021_2023"] - comparacion_2_3["frio_2022_2023"]).abs()

print("\n2. ¿COINCIDE EL TOTAL DIARIO DE 'Frio (Kw)' EN LAS FECHAS SOLAPADAS?")
print("  Solapamiento 2020_2022 vs 2021_2023:")
print("    - Fechas donde coinciden exactamente:", (diferencia_1_2 == 0).sum())
print("    - Fechas donde difieren:", (diferencia_1_2 > 0).sum())
print("    - Fechas donde a alguno le falta el registro de 23:59 (NaN):")
print(comparacion_1_2[comparacion_1_2.isna().any(axis=1)])

print("\n  Solapamiento 2021_2023 vs 2022_2023:")
print("    - Fechas donde coinciden exactamente:", (diferencia_2_3 == 0).sum())
print("    - Fechas donde difieren:", (diferencia_2_3 > 0).sum())
print("    - Fechas donde a alguno le falta el registro de 23:59 (NaN):")
print(comparacion_2_3[comparacion_2_3.isna().any(axis=1)])

1. CANTIDAD DE FECHAS SOLAPADAS ENTRE ARCHIVOS:
  - Entre 2020_2022 y 2021_2023: 254 fechas
  - Entre 2021_2023 y 2022_2023: 244 fechas
  - Entre 2020_2022 y 2022_2023: 0 fechas

2. ¿COINCIDE EL TOTAL DIARIO DE 'Frio (Kw)' EN LAS FECHAS SOLAPADAS?
  Solapamiento 2020_2022 vs 2021_2023:
    - Fechas donde coinciden exactamente: 246
    - Fechas donde difieren: 5
    - Fechas donde a alguno le falta el registro de 23:59 (NaN):
            frio_2020_2022  frio_2021_2023
DIA                                       
2021-07-13             NaN             NaN
2022-03-02             NaN             NaN
2022-03-16             NaN         24066.0

  Solapamiento 2021_2023 vs 2022_2023:
    - Fechas donde coinciden exactamente: 243
    - Fechas donde difieren: 0
    - Fechas donde a alguno le falta el registro de 23:59 (NaN):
            frio_2021_2023  frio_2022_2023
DIA                                       
2023-03-06             NaN         19199.0


In [7]:
# Mostramos las 5 fechas donde el archivo 2020_2022 y el 2021_2023 tienen valores distintos
print("1. LAS 5 FECHAS DONDE DIFIEREN 2020_2022 Y 2021_2023:")
print(comparacion_1_2[diferencia_1_2 > 0])

# Medimos la diferencia entre las 23:00 y las 23:59 (usando el primer archivo como muestra)
df_horas = df_ee_1.dropna(subset=["DIA"]).copy()
df_horas["HORA_texto"] = df_horas["HORA"].astype(str).str.strip()
df_horas["Frio (Kw)"] = pd.to_numeric(df_horas["Frio (Kw)"], errors="coerce")

frio_2300 = df_horas[df_horas["HORA_texto"].str.startswith("23:00")].groupby("DIA")["Frio (Kw)"].first()
frio_2359 = df_horas[df_horas["HORA_texto"].str.startswith("23:59")].groupby("DIA")["Frio (Kw)"].first()

dif_horas = (frio_2359 - frio_2300).dropna()
print("\n2. DIFERENCIA ENTRE LAS 23:59 Y LAS 23:00:")
print("  - Diferencia promedio:", round(dif_horas.mean(), 2), "kW")
print("  - Diferencia mediana: ", round(dif_horas.median(), 2), "kW")

#Revisamos las primeras columnas de cada una de las 15 hojas comunes
print("\n3. COLUMNAS INICIALES DE CADA HOJA EN EL EXCEL:")
for nombre_hoja in hojas_2020_2022:
    df_muestra = pd.read_excel(ruta_2020_2022, sheet_name=nombre_hoja, nrows=2)
    print(f"  - {nombre_hoja}: {len(df_muestra.columns)} columnas -> {list(df_muestra.columns[:4])}")

1. LAS 5 FECHAS DONDE DIFIEREN 2020_2022 Y 2021_2023:
            frio_2020_2022  frio_2021_2023
DIA                                       
2021-07-28       2900770.0         30764.0
2021-08-25       2907122.0         37122.0
2021-11-28       6155928.0         25719.0
2022-02-02         52655.0         19887.0
2022-02-18       8861470.0         21470.0

2. DIFERENCIA ENTRE LAS 23:59 Y LAS 23:00:
  - Diferencia promedio: 5726.65 kW
  - Diferencia mediana:  970.0 kW

3. COLUMNAS INICIALES DE CADA HOJA EN EL EXCEL:
  - Consolidado KPI: 62 columnas -> ['DIA', 'HORA', 'EE Planta / Hl', 'EE Elaboracion / Hl']
  - Consolidado Produccion: 12 columnas -> ['DIA', 'HORA', 'Hl de Mosto', 'Hl Cerveza Cocina']
  - Totalizadores Produccion: 40 columnas -> ['DIA', 'HORA', 'HL Mosto Budweiser', 'HL Mosto Tecate']
  - Consolidado EE: 21 columnas -> ['DIA', 'HORA', 'Planta (Kw)', 'Elaboracion (Kw)']
  - Totalizadores Energia: 53 columnas -> ['DIA', 'HORA', 'KW Gral Planta', 'KW Trafo 4']
  - Consolidado 

Primero observamos que en el archivo viejo (2020_2022), hay 4 fechas que estan en el orden de millones como se menciono en la consigna sobre los errores de registros, en los archivos mas nuevos eso se corrigio.

Utilizaremos 23:59 como cierre diario porque representa el acumulado íntegro de las 24 horas de la jornada.

Regla de desempate en fechas que coinciden:
- Cuando una fecha aparezca en más de un archivo, se prioriza siempre el archivo más reciente (2022_2023 > 2021_2023 > 2020_2022).
- El archivo más reciente contiene las correcciones de los valores corruptos de millones y posee la jornada completa en las fechas límite (evita los registros incompletos del final de los archivos más antiguos).

### Limpieza y aplicacion de la regla de desempate que definimos

In [8]:
#  Funcion para recortar las fechas con mas de 25 registros a su primera jornada real (25 filas)
def limpiar_jornadas_apiladas(df):
    # Quitamos filas sin fecha valida
    df_limpio = df.dropna(subset=["DIA"]).copy()
    
    # Aseguramos que HORA sea texto limpio para poder ordenar sin errores de tipo
    df_limpio["HORA"] = df_limpio["HORA"].astype(str).str.strip()
    
    # Por cada fecha ('DIA'), nos quedamos unicamente con las primeras 25 filas (.head(25))
    df_limpio = df_limpio.groupby("DIA").head(25)
    return df_limpio

# Limpiamos cada archivo por separado antes de unirlos
df_ee_1_limpio = limpiar_jornadas_apiladas(df_ee_1)
df_ee_2_limpio = limpiar_jornadas_apiladas(df_ee_2)
df_ee_3_limpio = limpiar_jornadas_apiladas(df_ee_3)

#  Aplicamos la regla de desempate: priorizar siempre el archivo mas reciente
fechas_archivo_3 = set(df_ee_3_limpio["DIA"].unique())
fechas_archivo_2 = set(df_ee_2_limpio["DIA"].unique())

# Del archivo 3 (mas reciente: 2022_2023) tomamos todas sus fechas
parte_3 = df_ee_3_limpio.copy()

# Del archivo 2 (2021_2023) tomamos solo las fechas que NO esten en el archivo 3
parte_2 = df_ee_2_limpio[~df_ee_2_limpio["DIA"].isin(fechas_archivo_3)].copy()

# Del archivo 1 (2020_2022) tomamos solo las fechas que NO esten ni en el 3 ni en el 2
fechas_ya_cubiertas = fechas_archivo_3.union(fechas_archivo_2)
parte_1 = df_ee_1_limpio[~df_ee_1_limpio["DIA"].isin(fechas_ya_cubiertas)].copy()

#  Unimos las tres partes y ordenamos cronologicamente por DIA y HORA
df_ee_unificado = pd.concat([parte_1, parte_2, parte_3], ignore_index=True)
df_ee_unificado = df_ee_unificado.sort_values(by=["DIA", "HORA"]).reset_index(drop=True)

# Verificaciones pedidas por la consigna
filas_por_dia = df_ee_unificado.groupby("DIA").size()
dias_con_mas_de_25 = filas_por_dia[filas_por_dia > 25]

print("VERIFICACION TRAS APLICAR REGLA DE DESEMPATE Y LIMPIEZA:")
print("  - Filas totales del dataset unificado:", len(df_ee_unificado))
print("  - Fecha inicial:", df_ee_unificado["DIA"].min().date(), "| Fecha final:", df_ee_unificado["DIA"].max().date())
print("  - Cantidad total de dias unicos:", df_ee_unificado["DIA"].nunique())
print("  - Dias con mas de 25 registros:", len(dias_con_mas_de_25))
print("  - Distribucion de registros por dia (filas_en_el_dia: cantidad_de_dias):")
print("   ", filas_por_dia.value_counts().to_dict())

VERIFICACION TRAS APLICAR REGLA DE DESEMPATE Y LIMPIEZA:
  - Filas totales del dataset unificado: 29698
  - Fecha inicial: 2020-07-01 | Fecha final: 2023-10-26
  - Cantidad total de dias unicos: 1190
  - Dias con mas de 25 registros: 0
  - Distribucion de registros por dia (filas_en_el_dia: cantidad_de_dias):
    {25: 1177, 24: 5, 23: 4, 16: 1, 20: 1, 17: 1, 8: 1}


Bueno se filtraron aprox 14 mil filas y tiene sentido.... entre los 3 archivos sumaban algo de 43 mil filas y si multiplicamos las filas de las fechas repetidas las cuentitas cierran.



### con el dataset ya unificado, realizamos los puntos del incios 1.1

### Checksum del dataset unificado , dimensiones y variables

In [9]:
import json

# Guardamos el dataset unificado inicial y registramos su checksum SHA-256
ruta_unificado = RAW_DATA_DIR / "dataset_unificado_inicial.csv"
df_ee_unificado.to_csv(ruta_unificado, index=False)

# Calculamos el hash SHA-256 leyendo por bloques de 1 MiB con la funcion huella()
hash_unificado = huella(ruta_unificado, algoritmo="sha256")

# Actualizamos el archivo data/checksums.json agregando el dataset unificado inicial
ruta_checksums = DATA_DIR / "checksums.json"
with open(ruta_checksums, "r", encoding="utf-8") as archivo_json:
    registro_checksums = json.load(archivo_json)

registro_checksums["dataset_unificado_inicial.csv"] = hash_unificado

with open(ruta_checksums, "w", encoding="utf-8") as archivo_json:
    json.dump(registro_checksums, archivo_json, indent=4)

print("1. CHECKSUM SHA-256 DEL DATASET UNIFICADO INICIAL:")
print("  - Archivo:", ruta_unificado.name)
print("  - SHA-256:", hash_unificado)

# Dimensiones del dataset unificado
print("\n2. DIMENSIONES DEL DATASET UNIFICADO:")
print("  - Filas:", df_ee_unificado.shape[0], "| Columnas:", df_ee_unificado.shape[1])

# Tipos de variables, rangos de valores (minimo y maximo) y valores faltantes
# Identificamos las columnas de mediciones (todas menos DIA y HORA)
columnas_mediciones = [col for col in df_ee_unificado.columns if col not in ["DIA", "HORA"]]

# Guardamos el tipo de dato original antes de convertir a numero
tipos_originales = df_ee_unificado[columnas_mediciones].dtypes

# Convertimos todas las columnas de mediciones a numerico (errores como #VALUE! pasan a ser NaN)
for col in columnas_mediciones:
    df_ee_unificado[col] = pd.to_numeric(df_ee_unificado[col], errors="coerce")

# Armamos una tabla resumen clara con Tipos, Minimo, Maximo y Valores Faltantes
resumen_variables = pd.DataFrame({
    "tipo_original": tipos_originales,
    "tipo_numerico": df_ee_unificado[columnas_mediciones].dtypes,
    "valor_minimo": df_ee_unificado[columnas_mediciones].min(),
    "valor_maximo": df_ee_unificado[columnas_mediciones].max(),
    "faltantes_cantidad": df_ee_unificado[columnas_mediciones].isna().sum(),
    "faltantes_porcentaje": (df_ee_unificado[columnas_mediciones].isna().mean() * 100).round(2)
})

print("\n3. TIPOS DE VARIABLES, RANGOS DE VALORES Y VALORES FALTANTES:")
print(resumen_variables.to_string())

1. CHECKSUM SHA-256 DEL DATASET UNIFICADO INICIAL:
  - Archivo: dataset_unificado_inicial.csv
  - SHA-256: 41e1d971dbb16c5d5a8068e63937c116d2975582794c8dd2e5a0ee3cacfcc9a4

2. DIMENSIONES DEL DATASET UNIFICADO:
  - Filas: 29698 | Columnas: 21

3. TIPOS DE VARIABLES, RANGOS DE VALORES Y VALORES FALTANTES:
                     tipo_original tipo_numerico  valor_minimo  valor_maximo  faltantes_cantidad  faltantes_porcentaje
Planta (Kw)                float64       float64 -1.211471e+06  9.974667e+06                   0                  0.00
Elaboracion (Kw)           float64       float64 -3.913000e+03  2.268656e+06                   0                  0.00
Bodega (Kw)                float64       float64 -2.252337e+06  1.787988e+04                   0                  0.00
Cocina (Kw)                float64       float64  0.000000e+00  2.262043e+06                   0                  0.00
Envasado (Kw)              float64       float64  0.000000e+00  3.277128e+16                   0   

ya notamos que hay valores que no cumplen el rango... hay valores exagerados (billones) y tambien valores de kw negativos, lo que es imposible.

### deteccion y cuantificacion de registros invalidos

In [10]:
print("DETECCION DE VALORES ATIPICOS Y REGISTROS INVALIDOS")
print("=" * 80)

# DIAS INCOMPLETOS (menos de 24 lecturas horarias)
lecturas_por_dia = df_ee_unificado.groupby("DIA").size()
dias_incompletos = lecturas_por_dia[lecturas_por_dia < 24]

print(f"\n1. DIAS CON MENOS DE 24 LECTURAS HORARIAS: {len(dias_incompletos)} dias")
for fecha, cantidad in dias_incompletos.items():
    print(f"   - {fecha.strftime('%Y-%m-%d')}: {cantidad} lecturas horarias")

#  VALORES IMPOSIBLES EN 'Frio (Kw)' (orden de los millones o negativos)
# Segun la documentacion, el rango tipico diario de Frio (Kw) es de 0 a 35.000 kW
filas_frio_millones = df_ee_unificado[df_ee_unificado["Frio (Kw)"] >= 1_000_000]
filas_frio_excesivo = df_ee_unificado[df_ee_unificado["Frio (Kw)"] > 100_000]
filas_frio_negativo = df_ee_unificado[df_ee_unificado["Frio (Kw)"] < 0]

dias_con_millones_frio = filas_frio_millones["DIA"].unique()
dias_con_excesivo_frio = filas_frio_excesivo["DIA"].unique()

print("\n2. VALORES IMPOSIBLES / EXTREMOS EN 'Frio (Kw)':")
print(f"   - Filas con Frio (Kw) >= 1.000.000 kW (orden de millones): {len(filas_frio_millones)} filas en {len(dias_con_millones_frio)} dias")
for fecha in dias_con_millones_frio:
    max_dia = df_ee_unificado[df_ee_unificado["DIA"] == fecha]["Frio (Kw)"].max()
    print(f"       * { pd.Timestamp(fecha).strftime('%Y-%m-%d') }: valor maximo = {max_dia:,.2f} kW")

print(f"   - Filas con Frio (Kw) > 100.000 kW: {len(filas_frio_excesivo)} filas en {len(dias_con_excesivo_frio)} dias")
print(f"   - Filas con Frio (Kw) negativo (< 0): {len(filas_frio_negativo)} filas")

# Tambien verificamos cuantas filas tienen valores >= 1.000.000 o < 0 en CUALQUIERA de las columnas de la hoja
mascara_millones_cualquier_col = (df_ee_unificado[columnas_mediciones] >= 1_000_000).any(axis=1)
mascara_negativos_cualquier_col = (df_ee_unificado[columnas_mediciones] < 0).any(axis=1)
print(f"   - Filas con valores >= 1.000.000 en alguna columna de Consolidado EE: {mascara_millones_cualquier_col.sum()} filas")
print(f"   - Filas con valores negativos (< 0) en alguna columna de Consolidado EE: {mascara_negativos_cualquier_col.sum()} filas")

#  DIAS EN QUE EL TOTALIZADOR NO SE REINICIO A MEDIANOCHE (00:00:00)
# A las 00:00:00 el totalizador debe reiniciarse a 0
registros_medianoche = df_ee_unificado[df_ee_unificado["HORA"].str.startswith("00:00")].copy()
dias_sin_reinicio_frio = registros_medianoche[registros_medianoche["Frio (Kw)"] > 0]

print(f"\n3. DIAS EN QUE EL TOTALIZADOR 'Frio (Kw)' NO SE REINICIO A 0 A MEDIANOCHE (00:00):")
print(f"   - Cantidad de dias con Frio (Kw) > 0 a las 00:00: {len(dias_sin_reinicio_frio)} dias")
if len(dias_sin_reinicio_frio) > 0:
    print(dias_sin_reinicio_frio[["DIA", "HORA", "Planta (Kw)", "Frio (Kw)"]].head(15).to_string(index=False))

DETECCION DE VALORES ATIPICOS Y REGISTROS INVALIDOS

1. DIAS CON MENOS DE 24 LECTURAS HORARIAS: 8 dias
   - 2021-07-01: 23 lecturas horarias
   - 2021-07-13: 23 lecturas horarias
   - 2021-11-01: 23 lecturas horarias
   - 2023-01-17: 16 lecturas horarias
   - 2023-04-13: 20 lecturas horarias
   - 2023-04-19: 17 lecturas horarias
   - 2023-07-01: 23 lecturas horarias
   - 2023-10-26: 8 lecturas horarias

2. VALORES IMPOSIBLES / EXTREMOS EN 'Frio (Kw)':
   - Filas con Frio (Kw) >= 1.000.000 kW (orden de millones): 3 filas en 3 dias
       * 2021-01-31: valor maximo = 32,771,277,123,578,720.00 kW
       * 2021-04-22: valor maximo = 32,771,276,800,018,260.00 kW
       * 2021-06-12: valor maximo = 4,111,695.00 kW
   - Filas con Frio (Kw) > 100.000 kW: 3 filas en 3 dias
   - Filas con Frio (Kw) negativo (< 0): 0 filas
   - Filas con valores >= 1.000.000 en alguna columna de Consolidado EE: 81 filas
   - Filas con valores negativos (< 0) en alguna columna de Consolidado EE: 1822 filas

3. DIA

# Fase 1.2 Analisis temporal

Si intentamos graficar la evolucion o calcular el promedio por dia dejando los dias que tienen valores exagerados, dias incompletos o el dia sin reinicio, los graficos seran un desastre. Vamos a revisar las horas exacta en que ocurrieron los  valores de millones/billones (para verificar que ocurran en la fila final). Filtraremos eel cierre diario a las 23:59.

In [11]:
# Verificamos en que hora exacta ocurren los 3 valores de millones/billones en 'Frio (Kw)'
print("1. DETALLE DE LAS 3 FILAS CON VALORES IMPOSIBLES EN 'Frio (Kw)':")
print(filas_frio_millones[["DIA", "HORA", "Planta (Kw)", "Frio (Kw)"]].to_string(index=False))

# Construimos el DataFrame diario quedandonos con el cierre de las 23:59 (Paso 1 de la consigna)
# Primero identificamos las fechas invalidas detectadas en la Seccion 1.1:
fechas_incompletas = set(dias_incompletos.index)
fechas_sin_reinicio = set(
    registros_medianoche[registros_medianoche["Frio (Kw)"] > 100]["DIA"]
)

# Filtramos las filas de cierre diario (23:59)
df_diario_bruto = df_ee_unificado[df_ee_unificado["HORA"].str.startswith("23:59")].copy()

# Quitamos los dias incompletos (< 24 lecturas), el dia sin reinicio (2020-07-11)
# y los 3 dias con valores imposibles (> 100.000 kW en Frio)
mascara_dias_validos = (
    (~df_diario_bruto["DIA"].isin(fechas_incompletas)) &
    (~df_diario_bruto["DIA"].isin(fechas_sin_reinicio)) &
    (df_diario_bruto["Frio (Kw)"] >= 0) &
    (df_diario_bruto["Frio (Kw)"] <= 100_000)
)

df_diario_eda = df_diario_bruto[mascara_dias_validos].sort_values("DIA").reset_index(drop=True)

print("\n2. RESUMEN DEL DATASET DIARIO PARA EL ANALISIS EXPLORATORIO (23:59):")
print("  - Total de dias con registro 23:59 antes de filtrar invalidos:", len(df_diario_bruto))
print("  - Total de dias validos para analizar:", len(df_diario_eda))
print("\n3. ESTADISTICAS DESCRIPTIVAS REALES DE 'Frio (Kw)' DIARIO:")
print(df_diario_eda["Frio (Kw)"].describe().round(2))

1. DETALLE DE LAS 3 FILAS CON VALORES IMPOSIBLES EN 'Frio (Kw)':
       DIA     HORA  Planta (Kw)    Frio (Kw)
2021-01-31 11:00:00     22979.10 3.277128e+16
2021-04-22 16:00:00     53019.60 3.277128e+16
2021-06-12 01:00:00      3769.23 4.111695e+06

2. RESUMEN DEL DATASET DIARIO PARA EL ANALISIS EXPLORATORIO (23:59):
  - Total de dias con registro 23:59 antes de filtrar invalidos: 1184
  - Total de dias validos para analizar: 1179

3. ESTADISTICAS DESCRIPTIVAS REALES DE 'Frio (Kw)' DIARIO:
count     1179.00
mean     23709.07
std       7720.40
min       2310.00
25%      17996.00
50%      23778.00
75%      29345.50
max      56128.00
Name: Frio (Kw), dtype: float64


In [12]:
#Filtramos los dias donde el totalizador diario de 'Frio (Kw)' supera los 40.000 kW, y vemos su hora

df_filtro_frio = df_diario_eda[df_diario_eda["Frio (Kw)"] > 40000].copy()
print("\n4. DIAS CON 'Frio (Kw)' DIARIO > 40.000 kW:")
print("  - Cantidad de dias:", len(df_filtro_frio))
print("  - Fechas y valores de 'Frio (Kw)' diario:")
print(df_filtro_frio[["DIA", "Frio (Kw)"]].sort_values("Frio (Kw)", ascending=False).to_string(index=False))




4. DIAS CON 'Frio (Kw)' DIARIO > 40.000 kW:
  - Cantidad de dias: 18
  - Fechas y valores de 'Frio (Kw)' diario:
       DIA  Frio (Kw)
2021-08-01    56128.0
2021-07-21    54681.0
2021-06-22    48046.0
2021-07-12    46001.0
2021-04-29    42919.0
2021-08-24    42604.0
2021-06-23    42005.0
2022-06-04    41901.0
2022-07-20    41206.0
2021-06-16    41203.0
2021-07-27    41148.0
2021-07-15    40979.0
2020-07-23    40886.0
2022-05-28    40881.0
2021-07-22    40720.0
2021-06-11    40693.0
2022-07-26    40648.0
2021-07-14    40329.0


In [13]:
df_horario = df_ee_unificado.sort_values(["DIA", "HORA"]).copy()

# Columna nueva con el consumo de cada hora (lectura actual - anterior, sin mezclar días)
df_horario["incremento_frio"] = df_horario.groupby("DIA")["Frio (Kw)"].diff()

# Mayor salto en una sola hora, por día
max_salto_por_dia = df_horario.groupby("DIA")["incremento_frio"].max()

# Nos quedamos con las horas de los días válidos (los 1179 de df_diario_eda), como df_diario_eda ya tiene los dias validos,
# sin los dias con problemas de reinicio, incompletos o con valores imposibles, podemos usarlo para filtrar el df_horario
dias_validos = df_diario_eda["DIA"]
df_horario_valido = df_horario[df_horario["DIA"].isin(dias_validos)]

df_horario_valido["incremento_frio"].describe(percentiles=[0.5, 0.9, 0.99, 0.999])


count    2.829200e+04
mean     9.880176e+02
std      3.896720e+14
min     -3.277128e+16
50%      9.700000e+02
90%      1.488000e+03
99%      1.924000e+03
99.9%    6.062557e+03
max      3.277128e+16
Name: incremento_frio, dtype: float64

Observamos como existen dentro de los datos aún valores de Frio los cuales no corresponden a un valor de medicion correcto, tenemos un minimo -3.277128e+16 kw y un maximo 3.277128e+16 Kw en el incremento de frio hora por hora, si bien el filtro que saca los valores imposibles de frio funciona, solamente lo hace con el acumulado hasta la hora 23:59, lo cual esta bien, pero con estos datos tenemos distintos escenarios los cuales debemos tener en cuenta

In [23]:
# Filas con saltos imposibles (del orden de los millones o más) dentro de los días válidos
filas_raras = df_horario_valido[df_horario_valido["incremento_frio"].abs() > 1_000_000]
print(filas_raras[["DIA", "HORA", "Frio (Kw)", "incremento_frio"]].to_string(index=False))
print("\nCantidad de filas:", len(filas_raras), "| Días afectados:", filas_raras["DIA"].nunique())

       DIA     HORA    Frio (Kw)  incremento_frio
2021-01-31 11:00:00 3.277128e+16     3.277128e+16
2021-01-31 12:00:00 7.137000e+03    -3.277128e+16
2021-04-22 16:00:00 3.277128e+16     3.277128e+16
2021-04-22 17:00:00 1.968600e+04    -3.277128e+16
2021-06-12 01:00:00 4.111695e+06     4.111695e+06
2021-06-12 02:00:00 3.558000e+03    -4.108137e+06

Cantidad de filas: 6 | Días afectados: 3


In [22]:
# Para cada lectura errónea: la hora anterior, la errónea y las dos siguientes
df_h = df_horario_valido.reset_index(drop=True)
posiciones_error = df_h.index[df_h["incremento_frio"] > 1_000_000]

for i in posiciones_error:
    tramo = df_h.loc[i - 1 : i + 2, ["DIA", "HORA", "Frio (Kw)", "incremento_frio"]]
    print(tramo.to_string(index=False))

    # Si el medidor no perdió la cuenta, (hora siguiente - hora anterior) debería ser
    # el consumo normal de 2 horas (~2.000 kW)
    consumo_2h = df_h.loc[i + 1, "Frio (Kw)"] - df_h.loc[i - 1, "Frio (Kw)"]
    print(f"   -> Consumo entre la hora anterior y la siguiente (2 h): {consumo_2h:,.0f} kW\n")

       DIA     HORA    Frio (Kw)  incremento_frio
2021-01-31 10:00:00 5.747000e+03     6.670000e+02
2021-01-31 11:00:00 3.277128e+16     3.277128e+16
2021-01-31 12:00:00 7.137000e+03    -3.277128e+16
2021-01-31 13:00:00 7.730000e+03     5.930000e+02
   -> Consumo entre la hora anterior y la siguiente (2 h): 1,390 kW

       DIA     HORA    Frio (Kw)  incremento_frio
2021-04-22 15:00:00 1.730700e+04     9.330000e+02
2021-04-22 16:00:00 3.277128e+16     3.277128e+16
2021-04-22 17:00:00 1.968600e+04    -3.277128e+16
2021-04-22 18:00:00 2.087300e+04     1.187000e+03
   -> Consumo entre la hora anterior y la siguiente (2 h): 2,379 kW

       DIA     HORA  Frio (Kw)  incremento_frio
2021-06-12 00:00:00        0.0              NaN
2021-06-12 01:00:00  4111695.0        4111695.0
2021-06-12 02:00:00     3558.0       -4108137.0
2021-06-12 03:00:00     4796.0           1238.0
   -> Consumo entre la hora anterior y la siguiente (2 h): 3,558 kW



En el incremento horario aparecen 6 saltos imposibles (±3,28·10¹⁶ kW y ±4,1·10⁶ kW) que corresponden a 3 lecturas erróneas del medidor: 2021-01-31 a las 11:00, 2021-04-22 a las 16:00 y 2021-06-12 a la 01:00. Estas lecturas son las mismas que encontramos en la seccion 1.1.

En los tres casos la lectura de la hora siguiente retoma la progresión normal: entre la hora anterior y la posterior al error el acumulado aumenta 1.390 kW (2021-01-31), 2.379 kW (2021-04-22) y 3.558 kW (2021-06-12), valores compatibles con dos horas de consumo normal en cada época del año (~700 kW/h en enero, ~1.200 kW/h en abril, ~1.800 kW/h en junio, por debajo del percentil 99 horario). Es decir, el medidor registró un valor erróneo puntual pero no perdió la cuenta: el cierre de las 23:59 es correcto.

Estos días no los excluyó el filtro de la celda anterior porque las condiciones `>= 0` y `<= 100.000` se aplican solo sobre la fila de cierre (23:59).

Decision: mantenemos los 3 días en `df_diario_eda`, ya que su total diario es válido. Para los análisis sobre datos horarios (como la distribución de incrementos) excluimos los saltos mayores a 1.000.000 kW.

In [25]:
# Distribución del consumo horario sin las 3 lecturas erróneas
incremento_normal = df_horario_valido.loc[df_horario_valido["incremento_frio"].abs() < 1_000_000, "incremento_frio"]
print(incremento_normal.describe(percentiles=[0.5, 0.9, 0.99, 0.999]).round(1))

# Ahora veremos si hay horas en las que el acumulado baja (no debería pasar en un totalizador)
bajadas = df_horario_valido[(df_horario_valido["incremento_frio"] < 0) & (df_horario_valido["incremento_frio"].abs() < 1_000_000)]
print("\nHoras con incremento negativo:", len(bajadas), "| Días afectados:", bajadas["DIA"].nunique())

count    28286.0
mean       988.0
std        752.3
min     -31131.0
50%        970.0
90%       1487.0
99%       1923.2
99.9%     5153.4
max      33521.0
Name: incremento_frio, dtype: float64

Horas con incremento negativo: 19 | Días afectados: 19


In [26]:
# Para cada bajada: el incremento de la hora anterior y el cierre del día
df_h["incremento_anterior"] = df_h.groupby("DIA")["incremento_frio"].shift(1)

bajadas = df_h[(df_h["incremento_frio"] < 0) & (df_h["incremento_frio"].abs() < 1_000_000)]
bajadas = bajadas.merge(
    df_diario_eda[["DIA", "Frio (Kw)"]].rename(columns={"Frio (Kw)": "cierre_23_59"}),
    on="DIA",
)
print(bajadas[["DIA", "HORA", "Frio (Kw)", "incremento_frio", "incremento_anterior", "cierre_23_59"]].to_string(index=False))

       DIA     HORA  Frio (Kw)  incremento_frio  incremento_anterior  cierre_23_59
2020-09-11 13:00:00    16733.0         -28674.0               1121.0       29293.0
2021-01-03 18:00:00     3350.0            -10.0                  0.0        4925.0
2021-01-21 06:00:00     2344.0            -13.0                139.0       16699.0
2021-03-18 07:00:00     8005.0         -27138.0              29166.0       24261.0
2021-03-19 13:00:00    13725.0         -26003.0              27976.0       23964.0
2021-04-04 18:00:00    17526.0         -31131.0               1374.0       24715.0
2021-04-21 12:00:00    16281.0          -5405.0               8139.0       31212.0
2021-04-25 12:00:00    13308.0          -8660.0               5710.0       27720.0
2021-04-30 06:00:00     8655.0          -8446.0              11518.0       35657.0
2021-06-27 10:00:00     7099.0          -2128.0               1025.0       31198.0
2021-07-11 05:00:00     7518.0          -8790.0              11453.0       37846.0
2021

Dentro de los 1.179 días válidos hay 19 días en los que el acumulado de Frio baja en alguna hora, algo que no debería pasar en un totalizador. Los clasificamos en cuatro escenarios:

-Observamos que en 10 dias, por ejemplo 2021-03-18: +29166 kW y a la hora siguiente -27138 kW, tienen un incremento_anterior parecido al incremento_frio, estos son errores que vuelven en la hora siguiente, subiendo y bajando un tamaño parecido, esto hace que el cierre quede bien, o en su defecto un poco inflado como mucho entre 1000 o 3000 Kw de mas

-Luego observamos como en los dias 2020-09-11, 2021-04-04 el incremento_frio es mucho mas grande que incremento_anterior, por ejemplo el dia 2020-09-11: 1121.0 y a la hora siguiente es de: -28674.0, esto se debe a un error que vuelve a la normalidad varias horas despues, lo cual hace que el cierre quede bien.

-los dias 2021-01-21 y 2021-07-14, con bajadas de 13 y 40 kW se deben al ruido, por lo que son despreciables.

-Para los dias 2021-06-27, 2021-08-03, 2022-04-16 y 2022-07-03, con bajadas de 2000 a 7000 kW sin subida en la hora anterior, no se revisaron uno por uno y los evaluaremos al final de la Fase 2.

-Tenemos el caso del dia 2021-01-03, veamos que ocurre para este dia:

In [27]:
# Día 2021-01-03 hora por hora: acumulado e incremento
dia = df_horario[df_horario["DIA"] == "2021-01-03"]
print(dia[["HORA", "Frio (Kw)", "incremento_frio"]].to_string(index=False))

# Cuántas horas seguidas el contador no sumó nada
horas_sin_sumar = (dia["incremento_frio"].abs() <= 10).sum()
print(f"\nHoras con incremento ~0 (|incremento| <= 10 kW): {horas_sin_sumar}")

# Comparación con un día normal de enero
media_enero = df_diario_eda[df_diario_eda["DIA"].dt.month == 1]["Frio (Kw)"].mean()
cierre = dia["Frio (Kw)"].iloc[-1]
print(f"Cierre 23:59 del 2021-01-03: {cierre:,.0f} kW")
print(f"Media de los cierres de enero: {media_enero:,.0f} kW  ->  el día queda en el {cierre / media_enero:.0%} de lo normal")

    HORA  Frio (Kw)  incremento_frio
00:00:00        0.0              NaN
01:00:00      423.0            423.0
02:00:00      811.0            388.0
03:00:00     1186.0            375.0
04:00:00     1782.0            596.0
05:00:00     2336.0            554.0
06:00:00     2670.0            334.0
07:00:00     3013.0            343.0
08:00:00     3356.0            343.0
09:00:00     3360.0              4.0
10:00:00     3360.0              0.0
11:00:00     3360.0              0.0
12:00:00     3360.0              0.0
13:00:01     3360.0              0.0
14:00:00     3360.0              0.0
15:00:00     3360.0              0.0
16:00:00     3360.0              0.0
17:00:00     3360.0              0.0
18:00:00     3350.0            -10.0
19:00:00     3642.0            292.0
20:00:00     3849.0            207.0
21:00:00     4088.0            239.0
22:00:00     4347.0            259.0
23:00:00     4618.0            271.0
23:59:00     4925.0            307.0

Horas con incremento ~0 (|incremento|

Efectivamente estamos en un nuevo caso donde no podemos saber si el sistema estuvo apagado o si falló el medidor, pero la segunda opción es más probable: la refrigeración de una cervecería opera las 24 horas, y un totalizador detenido no retrocede (a las 18:00 el acumulado baja 10 kW). El medidor no conto durante 10 horas, desde las 09:00 y las 18:00. Observamos tambien como el cierre esta muy por debajo de lo normal, 36% de la media de enero lo cual señala que este dia esta mal.
Este dia se marca como invalido y lo excluiremos en la fase 2.1

In [28]:
# Mayor salto horario de cada día y a qué hora ocurrió (solo días válidos)
idx_max = df_h.groupby("DIA")["incremento_frio"].idxmax()
saltos = df_h.loc[idx_max, ["DIA", "HORA", "incremento_frio"]].rename(
    columns={"HORA": "hora_salto", "incremento_frio": "salto_maximo"}
)

# Cruzamos con los 18 días de más de 40.000 kW
dias_altos = df_filtro_frio[["DIA", "Frio (Kw)"]].merge(saltos, on="DIA")
dias_altos = dias_altos.sort_values("Frio (Kw)", ascending=False)
print(dias_altos.to_string(index=False))

       DIA  Frio (Kw) hora_salto  salto_maximo
2021-08-01    56128.0   22:00:00       27963.0
2021-07-21    54681.0   19:00:00       17689.0
2021-06-22    48046.0   15:00:00        2327.0
2021-07-12    46001.0   21:00:00        6597.0
2021-04-29    42919.0   14:00:00        4960.0
2021-08-24    42604.0   23:59:00       11161.0
2021-06-23    42005.0   19:00:00        2195.0
2022-06-04    41901.0   11:00:00        2065.0
2022-07-20    41206.0   18:00:00        1975.0
2021-06-16    41203.0   15:00:00        2201.0
2021-07-27    41148.0   23:59:00       14751.0
2021-07-15    40979.0   18:00:00        2018.0
2020-07-23    40886.0   06:00:00        2140.0
2022-05-28    40881.0   21:00:00        2081.0
2021-07-22    40720.0   04:00:00        2121.0
2021-06-11    40693.0   17:00:00        2120.0
2022-07-26    40648.0   21:00:00        2050.0
2021-07-14    40329.0   13:00:00        2359.0


Para analizar estos resultados tomaremos como umbral el percentil P99.9 = 5153 Kw, donde menos de una hora cada 1000 consume eso en condiciones normales, por encima de esto, una sola hora no lo explica. Con este criterio 13 de 18 dias son normales ya que sus saltos maximos van de 1975 a 2359 Kw, justo en la cola alta de una hora de verano, concluimos que son dias normales. 

Los 5 que superan el umbral:

-Dia: 2021-08-01; cierre: 56128; Horario: 22:00, salto: 27963, porcentaje del dia explicado por esta hora: 50%

-Dia: 2021-07-21; cierre: 54681; Horario: 19:00, salto: 17689, porcentaje del dia explicado por esta hora: 32%

-Dia: 2021-07-27; cierre: 41148; Horario: 23:59, salto: 14751, porcentaje del dia explicado por esta hora: 36%

-Dia: 2021-08-24; cierre: 42604; Horario: 23:59, salto: 11161, porcentaje del dia explicado por esta hora: 26%

-Dia: 2021-07-12; cierre: 46001; Horario: 21:00, salto: 6597, porcentaje del dia explicado por esta hora: 14%

Si comparamos estos 5 dias con los datos de la tabla de bajadas, hay cuatro de estos dias (2021-08-01, el 07-21, el 08-24 y el 07-12) que no aparecen, esto se debe a que subieron y nunca volvieron, por lo que corresponde a un escenario donde el incremento se ve como un salto enorme que se queda y no baja despues, provocando que el cierre quede inflado pero por debajo de 100000 Kw, por lo tanto el filtro no los detectara. El dia 2021-07-27 aparece pero con una bajada a las 18:00, antes del salto de las 23:59, así que tampoco lo compensa. 

Decision: junto con el 2021-01-03 (contador congelado), quedan 6 días marcados como inválidos para la Fase 2.1, en la cual evaluaremos excluirlos o corregir su cierre restando el salto. Ademas, el maximo de 56128 kW que observamos en la 1.1 corresponde a un error del medidor, el maximo real es el cierre del dia 2021-06-22: 48046 kW
